# v5.0 normalisation fix — confirmation & ROC

Confirms the resolution of the VDP P(NEO) wing-suppression, using the post-fix files from Hyak.

**Two bugs found (neither was the cloner):**
1. *Label-definition artifact* — the map trains on a broad catalog MBA (`1.7<=a<4.1, q>=1.3`) but the
   eval truth used narrow MBA (`2.0<a<3.3, e<0.3`) and dumped the belt edge into `other`. So apparent
   pure-NEO cells actually held `other`; VDP's low P there was largely correct. Fix: align eval labels.
2. *kNN density bleed* — the dense MBA core leaks density into zero-support NEO wing cells. Fix: a
   load-time **support-count mask** (`support_mask_min`) that zeros a population's density where its
   in-cell support is below threshold (NEO exempt — it is intentionally smoothed).

**Files:** `prob_maps_grid/*.npz` (667 maps; mask is load-time), `sorcha_comparison_v5_masked.parquet`
(`P_NEO_vdp` masked, `P_NEO_vdp_unmasked` kept). **Kernel:** `neofast_py310`.

**Headline:** the mask is correct but F1-neutral (0.808->0.809); the remaining VDP-digest2 gap is
intermediate-elongation velocity overlap (physics). VDP wins at the antisun (the NEOCP regime).

In [ ]:
import sys, re, glob
sys.path.insert(0, "adam_core_stub"); sys.path.insert(0, "src")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, precision_recall_curve
import velocity_density_pipeline_gmm as vdp
%matplotlib inline

df = pd.read_parquet("outputs/phase2/sorcha_comparison_v5_masked.parquet")

# training-aligned labels (match the map's catalog population definitions)
a, e, q = df.a_au.values, df.e.values, df.q_au.values
lab = np.full(len(df), "other", object)
lab[a > 30] = "TNO"
lab[(a > 4.7) & (a < 5.9) & (e < 0.3)] = "Trojan"
lab[(a >= 1.7) & (a < 4.1) & (q >= 1.3)] = "MBA_like"
lab[q < 1.3] = "NEO"
df["aligned"] = lab
y = (lab == "NEO")
df["absdlon"] = df.prob_map_file.map(
    lambda s: (lambda m: abs(int(m.group(1))) if m else 999)(re.search(r"dlon([+-]?\d+)", str(s))))

print("aligned counts:", {k: int((lab == k).sum()) for k in ['NEO','MBA_like','Trojan','TNO','other']})
print("NEO median P_vdp  masked:", round(df[y].P_NEO_vdp.median(), 3),
      " unmasked:", round(df[y].P_NEO_vdp_unmasked.median(), 3),
      " digest2:", round(df[y].P_NEO_d2.median(), 3))

## 1. Map P(NEO): the support mask fixes the wings

Load the antisun-centred grid map with and without `support_mask_min`. With the mask, pure-NEO wing
cells (where no MBA support exists) snap to P~1, while the MBA locus stays correctly dark.

In [ ]:
MAP = "prob_maps_grid/prob_maps_grid_dlon+000_lat+00.npz"   # antisun centre
pm0 = vdp.ProbMapSet.from_npz(MAP)                      # mask OFF
pm1 = vdp.ProbMapSet.from_npz(MAP, support_mask_min=1)  # mask ON

def clip08(pm, M, lim=0.8):
    s = np.abs(pm.x_grid) <= lim; i0, i1 = np.where(s)[0][[0, -1]]
    return M[i0:i1 + 1, i0:i1 + 1]
EXT = [-0.8, 0.8, -0.8, 0.8]

labs = ["18_20", "mag22", "mag24+"]
fig, ax = plt.subplots(len(labs), 2, figsize=(8, 3.6 * len(labs)))
for r, L in enumerate(labs):
    for c, (pm, t) in enumerate([(pm0, "mask OFF"), (pm1, "mask ON (support>=1)")]):
        M = clip08(pm, pm.get_probability_map(L, "NEO"))
        im = ax[r, c].imshow(M, origin="lower", extent=EXT, vmin=0, vmax=1, aspect="auto")
        ax[r, c].set_title(f"NEO {L}  [{t}]", fontsize=9)
        ax[r, c].axhline(0, color="w", lw=.4, ls=":"); ax[r, c].axvline(0, color="w", lw=.4, ls=":")
        plt.colorbar(im, ax=ax[r, c], fraction=.046)
        if c == 0: ax[r, c].set_ylabel(r"$v_\beta$ (deg/day)")
        if r == len(labs) - 1: ax[r, c].set_xlabel(r"$v_\lambda$ (deg/day)")
fig.suptitle("v5.0 antisun map P(NEO): support mask fixes the pure-NEO wings")
plt.tight_layout(); plt.show()

## 1b. S3M kNN vs new GMM grid — density & P(NEO) maps

The comparison from `sorcha_gmm_s3m_singleepoch_comparison.ipynb`, but with the **new v5.0 GMM grid map
(support mask ON)** on the right and an S3M-kNN antisun map (the broad reference that "got it right") on
the left. With the mask, the new GMM P(NEO) is now broad and bright across the NEO region like S3M,
with the MBA locus (vλ~-0.2) correctly excluded — vs. the old dim GMM.

In [ ]:
s3m = vdp.ProbMapSet.from_npz(sorted(glob.glob("prob_maps/*antisun*.npz"))[0])  # S3M-kNN reference
gmm = pm1                                                                       # new GMM grid (mask ON)
print(f"S3M ref center=({s3m.center_lon_deg:.0f},{s3m.center_lat_deg:.0f}) grid +/-{s3m.x_grid[-1]:.1f}")
print(f"GMM grid center=({gmm.center_lon_deg:.0f},{gmm.center_lat_deg:.0f}) grid +/-{gmm.x_grid[-1]:.1f}")
_LBL = {"18_20": "18-20", "mag22": "22-23", "mag24+": "24+"}

# --- P(NEO): S3M vs new GMM ---
fig, ax = plt.subplots(len(labs), 2, figsize=(8.5, 3.4 * len(labs)))
for r, b in enumerate(labs):
    for c, (pm, t) in enumerate([(s3m, "S3M kNN"), (gmm, "GMM grid (mask ON)")]):
        M = clip08(pm, pm.get_probability_map(b, "NEO"))
        im = ax[r, c].imshow(M, origin="lower", extent=EXT, vmin=0, vmax=1, aspect="auto")
        ax[r, c].set_title(f"NEO {_LBL[b]} mag  [{t}]", fontsize=9)
        ax[r, c].axhline(0, color="w", lw=.4, ls=":"); ax[r, c].axvline(0, color="w", lw=.4, ls=":")
        plt.colorbar(im, ax=ax[r, c], fraction=.046, label="P(NEO)" if c == 1 else None)
        if c == 0: ax[r, c].set_ylabel(r"$v_\beta$")
        if r == len(labs) - 1: ax[r, c].set_xlabel(r"$v_\lambda$")
fig.suptitle("P(NEO): S3M kNN  vs  new GMM grid (mask ON) — antisun, +/-0.8")
plt.tight_layout(); plt.show()

In [ ]:
# --- log density (6 rows): MBA x3 mag, NEO x3 mag, S3M vs new GMM ---
rows = [("MBA", "18_20"), ("MBA", "mag22"), ("MBA", "mag24+"),
        ("NEO", "18_20"), ("NEO", "mag22"), ("NEO", "mag24+")]
fig, ax = plt.subplots(len(rows), 2, figsize=(8, 3.0 * len(rows)), constrained_layout=True)
for r, (pop, b) in enumerate(rows):
    Ds = clip08(s3m, s3m.results[b]["density_maps_downweighted_raw"][pop])
    Dg = clip08(gmm, gmm.results[b]["density_maps_downweighted_raw"][pop])
    Ls = np.log10(np.where(Ds > 0, Ds, np.nan)); Lg = np.log10(np.where(Dg > 0, Dg, np.nan))
    vmax = np.nanmax([np.nanmax(Ls), np.nanmax(Lg)]); vmin = vmax - 5   # 5-decade range below peak
    im = None
    for c, (Lm, t) in enumerate([(Ls, "S3M kNN"), (Lg, "GMM")]):
        im = ax[r, c].imshow(Lm, origin="lower", extent=EXT, vmin=vmin, vmax=vmax, aspect="auto", cmap="viridis")
        ax[r, c].set_title(f"{pop} {_LBL[b]} mag  [{t}]", fontsize=9)
        ax[r, c].axhline(0, color="w", lw=.4, ls=":"); ax[r, c].axvline(0, color="w", lw=.4, ls=":")
        if c == 0: ax[r, c].set_ylabel(r"$v_\beta$")
        if r == len(rows) - 1: ax[r, c].set_xlabel(r"$v_\lambda$")
    fig.colorbar(im, ax=ax[r, :].tolist(), fraction=0.025, pad=0.02, label=r"$\log_{10} n_0$")
fig.suptitle("Log density: S3M kNN (left) vs new GMM grid (right) \u2014 antisun, +/-0.8", fontsize=12)
plt.show()

**Note — the MBA panels differ for reasons unrelated to GMM.** MBAs are K|M-cloned and kNN-density-
estimated in *both* maps (GMM only replaces the NEO cloner), so the left/right MBA differences are not a
GMM effect — they come from the two maps not being matched:

1. **MBA clone factor 10 (S3M ref) vs 5 (grid)** — ~2× fewer clones, so the kNN density is sampled more
   sparsely (biggest visual driver).
2. **Different epoch / antisun centre** (2025-05 vs 2026-01) — different MBAs are actually visible.
3. **S3M-only vs hybrid (S3M + MPCORB)** training catalog, with the hybrid using a broader MBA definition
   (1.7 ≤ a < 4.1).

The MBA velocity locus (vλ ≈ −0.2) is identical in structure; the differences are second-order and largest
at bright bins (fewer objects), converging by 24+. This is a consistency check, **not** a controlled
apples-to-apples comparison (unlike the old single-epoch notebook, which scored the *same* objects).

## 2. Calibration: masked vs unmasked vs digest2

On the **full representative** set (not the NEO-enriched subsample) all three are close to the diagonal;
the mask mainly recovers coverage in the wings (a completeness effect calibration can't see).

In [ ]:
def reliability(score, nbins=20):
    edges = np.linspace(0, 1, nbins + 1)
    idx = np.clip(np.digitize(np.clip(score, 0, 1), edges) - 1, 0, nbins - 1)
    xs, ys = [], []
    for b in range(nbins):
        m = idx == b
        if m.sum() >= 50: xs.append(score[m].mean()); ys.append(y[m].mean())
    return np.array(xs), np.array(ys)

plt.figure(figsize=(5.5, 5))
plt.plot([0, 1], [0, 1], "k--", lw=1, label="perfect")
for name, s in [("VDP masked", df.P_NEO_vdp), ("VDP unmasked", df.P_NEO_vdp_unmasked), ("digest2", df.P_NEO_d2)]:
    xs, ys = reliability(s.values); plt.plot(xs, ys, "o-", ms=4, label=name)
plt.xlabel("predicted P(NEO)"); plt.ylabel("actual NEO fraction")
plt.xlim(0, 1); plt.ylim(0, 1); plt.legend(); plt.title("v5.0 calibration (aligned labels)")
plt.tight_layout(); plt.show()

## 3. ROC, best-F1, per-population, and the antisun-distance breakdown

In [ ]:
def roc(name, s):
    s = s.to_numpy(); auc = roc_auc_score(y, s)
    p, r, t = precision_recall_curve(y, s)
    f1 = np.divide(2 * p * r, p + r, out=np.zeros_like(p), where=(p + r) > 0)
    bi = int(np.argmax(f1[:-1])); th = t[bi]; sel = s >= th
    compl = (sel & y).sum() / y.sum(); contam = (sel & ~y).sum() / max(sel.sum(), 1)
    print(f"  {name:16s} AUC={auc:.3f}  bestF1={f1[bi]:.3f}@{th:.3f}  compl={100*compl:.1f}%  contam={100*contam:.1f}%")
    return th

print("=== full-sky ROC (masked parquet, aligned labels, NEO=positive) ===")
tv = roc("VDP (masked)", df.P_NEO_vdp)
_  = roc("VDP (unmasked)", df.P_NEO_vdp_unmasked)
td = roc("digest2", df.P_NEO_d2)

print("\nper-population fraction above best-F1 threshold:")
print(f"{'population':>9} {'N':>9} {'VDP%':>7} {'d2%':>7}")
for pop in ['NEO','MBA_like','Trojan','TNO','other']:
    m = df.aligned == pop
    print(f"{pop:>9} {int(m.sum()):>9,} {100*(df.P_NEO_vdp[m]>=tv).mean():>7.1f} {100*(df.P_NEO_d2[m]>=td).mean():>7.1f}")

print("\nVDP vs digest2 best-F1 by |dlon| from antisun:")
for lo, hi in [(0,20),(20,40),(40,70),(70,110),(110,141)]:
    m = ((df.absdlon>=lo)&(df.absdlon<hi)).to_numpy()
    if y[m].sum() < 20: continue
    def bf(s): p,r,t=precision_recall_curve(y[m], s[m]); f=np.divide(2*p*r,p+r,out=np.zeros_like(p),where=(p+r)>0); return f[:-1].max()
    print(f"  {lo:>3}-{hi:<3} N={m.sum():>8,}  VDP F1={bf(df.P_NEO_vdp.values):.3f}  d2 F1={bf(df.P_NEO_d2.values):.3f}")

## 4. Resolution

- **Map P(NEO)** (§1, §1b): the support mask restores P~1 in pure-NEO wing cells, and the new GMM grid
  now matches the broad S3M-kNN NEO coverage — the correct map behaviour.
- **ROC** (§3): full-sky **VDP F1~0.809, AUC 0.880** vs **digest2 0.836 / 0.930**. The mask is F1-neutral
  (0.808->0.809) because the fixed wing cells hold few NEOs.
- **Antisun breakdown**: VDP **beats** digest2 at the antisun (0-20deg: 0.88 vs 0.85, the NEOCP discovery
  regime) and loses at intermediate elongation (40-110deg), where NEO and MBA velocities genuinely overlap
  — physics, not a bug.
- **Label alignment** (the methodological catch): using the catalog population boundaries collapses the
  spurious `other` class (23k -> 560), so the diagnostics use the same population definitions as the maps.

**Paper framing:** VDP is well-calibrated and strongest at opposition/antisun (where NEOs are discovered);
the off-antisun gap is elongation-dependent velocity overlap. The support mask is the correct map
behaviour; the label alignment is the diagnostic fix.

## 5. Direction comparison maps (new GMM grid) — Zeljko's figures

The direction-comparison grids from `paper_figures_ecliptic_strip.ipynb` §12, rebuilt with the
**new v5.0 GMM grid maps (support mask ON)** at four antisun-relative offsets (Δλ = 0°, −40°, −90°,
−120°; lat=0), clipped to ±0.8 deg/day to match the originals.

- **Density grid:** MBA and NEO densities, three magnitude bins, four directions.
- **NEO probability grid:** P(NEO) across directions — with the support mask the pure-NEO regions are
  bright (P≈1) while the non-NEO locus is dark, and you can see the velocity separation degrade as you
  move away from the antisun. This is the best visual proof of how the maps look now.

In [ ]:
# ===== Direction comparison maps (Zeljko's figures) -- new GMM grid, support mask ON =====
import matplotlib as mpl
_VEL_LIM = (-0.8, 0.8)

def _density_cmap():
    cm = mpl.colormaps["viridis"].copy(); cm.set_bad("0.88"); return cm

def _density_log_map(d):
    d = np.asarray(d, float); pos = np.isfinite(d) & (d > 0)
    lm = np.ma.masked_all(d.shape, float)
    if pos.any():
        lm[pos] = np.log10(d[pos]); vmax = float(np.nanmax(lm)); vmin = vmax - 5.0
    else:
        vmin, vmax = 0.0, 1.0
    return lm, vmin, vmax

def _annotate_no_support(ax, data):
    if np.ma.isMaskedArray(data) and np.ma.count(data) == 0:
        ax.text(0.5, 0.5, "no support", transform=ax.transAxes,
                ha="center", va="center", fontsize=10, color="0.25")

def _mbd(l):
    return {"18_20": "18-20", "mag20": "20-21", "mag21": "21-22", "mag22": "22-23", "mag23": "23-24", "mag24+": "24-25"}.get(l, l)

def _clip(pm, M, lim=0.8):
    s = np.abs(pm.x_grid) <= lim; i0, i1 = np.where(s)[0][[0, -1]]
    return M[i0:i1 + 1, i0:i1 + 1]

def _dir_map_data(pm, pop, b, mode):
    if mode == "density":
        return _density_log_map(_clip(pm, pm.results[b]["density_maps_downweighted_raw"][pop]))[0]
    return _clip(pm, pm.get_probability_map(b, pop))

# four antisun-relative direction maps from the new GMM grid (support mask ON)
DIRS = {"Anti-Sun (dlon=0)": "prob_maps_grid_dlon+000_lat+00.npz",
        "Anti-Sun -40 deg":  "prob_maps_grid_dlon-040_lat+00.npz",
        "Anti-Sun -90 deg":  "prob_maps_grid_dlon-090_lat+00.npz",
        "Anti-Sun -120 deg": "prob_maps_grid_dlon-120_lat+00.npz"}
dir_mapsets = {k: vdp.ProbMapSet.from_npz("prob_maps_grid/" + v, mask_radius_deg_per_day=float("inf"), support_mask_min=1)
               for k, v in DIRS.items()}

def plot_direction_grid(mapsets, pops, bins, mode, figsize):
    items = list(mapsets.items()); rows = [(p, b) for p in pops for b in bins]
    fig, ax = plt.subplots(len(rows), len(items), figsize=figsize,
                           sharex=True, sharey=True, constrained_layout=True)
    ax = np.atleast_2d(ax)
    for r, (pop, b) in enumerate(rows):
        rd = [_dir_map_data(pm, pop, b, mode) for _, pm in items]
        if mode == "density":
            sup = [d for d in rd if np.ma.count(d) > 0]
            vmax = max(float(np.nanmax(d)) for d in sup) if sup else 1.0
            vmin = (vmax - 5.0) if sup else 0.0; cbl = r"$\log_{10}\langle n_0\rangle$"
        else:
            vmin, vmax = 0.0, 1.0; cbl = "probability"
        im = None
        for c, ((dl, pm), d) in enumerate(zip(items, rd)):
            a = ax[r, c]
            im = a.imshow(d, origin="lower", extent=(*_VEL_LIM, *_VEL_LIM),
                          cmap=_density_cmap() if mode == "density" else "viridis",
                          vmin=vmin, vmax=vmax, interpolation="bilinear", aspect="equal")
            a.axvline(0, color="w", lw=.7, ls="--", alpha=.7); a.axhline(0, color="w", lw=.7, ls="--", alpha=.7)
            _annotate_no_support(a, d)
            if r == 0: a.set_title(f"{dl}\nlon={pm.center_lon_deg:.0f}, lat={pm.center_lat_deg:.0f}", fontsize=10)
            if c == 0: a.set_ylabel(f"{pop}: {_mbd(b)}\n" + r"$v_\beta$ (deg/day)", fontsize=12)
            if r == len(rows) - 1: a.set_xlabel(r"$v_\lambda$ (deg/day)", fontsize=12)
        fig.colorbar(im, ax=ax[r, :], label=cbl, shrink=0.8, pad=0.012)
    ttl = "Log density" if mode == "density" else "P(NEO), support mask ON"
    fig.suptitle(f"{ttl} direction comparison -- new GMM grid", fontsize=14)
    plt.show()

for k, pm in dir_mapsets.items():
    print(f"{k:20s} lon={pm.center_lon_deg:.0f} lat={pm.center_lat_deg:.0f}")

In [ ]:
# 6x4 density grid: MBA then NEO, three mag bins, four directions
plot_direction_grid(dir_mapsets, pops=("MBA", "NEO"),
                    bins=["mag21", "mag22", "mag24+"], mode="density", figsize=(14, 18))

In [ ]:
# 3x4 NEO probability grid (the key 'pure-NEO -> P~1' figure)
plot_direction_grid(dir_mapsets, pops=("NEO",),
                    bins=["mag21", "mag22", "mag24+"], mode="probability", figsize=(14.5, 9.2))

### Support mask OFF vs ON — MBA & NEO probability (antisun)

Reference view of the support-mask effect on **both** populations at the antisun. Mask ON confines
P(MBA) to the MBA velocity locus (zeroing it in the wings) and correspondingly raises P(NEO) to ~1
across the broad NEO region. Same magnitude bins (21-22, 22-23, 24-25) as the grids above.

In [ ]:
# ----- support mask OFF vs ON, MBA & NEO probability (antisun direction) -----
INF = float("inf")
ANTISUN = "prob_maps_grid/prob_maps_grid_dlon+000_lat+00.npz"
pm_off = vdp.ProbMapSet.from_npz(ANTISUN, mask_radius_deg_per_day=INF)                      # mask OFF
pm_on  = vdp.ProbMapSet.from_npz(ANTISUN, mask_radius_deg_per_day=INF, support_mask_min=1)  # mask ON
mc_bins = ["mag21", "mag22", "mag24+"]
mc_cols = [("MBA", pm_off, "MBA [mask OFF]"), ("MBA", pm_on, "MBA [mask ON]"),
           ("NEO", pm_off, "NEO [mask OFF]"), ("NEO", pm_on, "NEO [mask ON]")]
fig, ax = plt.subplots(len(mc_bins), 4, figsize=(13, 3.3 * len(mc_bins)),
                       sharex=True, sharey=True, constrained_layout=True)
for r, b in enumerate(mc_bins):
    im = None
    for c, (pop, pm, t) in enumerate(mc_cols):
        M = _clip(pm, pm.get_probability_map(b, pop))
        im = ax[r, c].imshow(M, origin="lower", extent=(*_VEL_LIM, *_VEL_LIM),
                             vmin=0, vmax=1, cmap="viridis", aspect="equal", interpolation="bilinear")
        ax[r, c].axvline(0, color="w", lw=.6, ls="--", alpha=.7); ax[r, c].axhline(0, color="w", lw=.6, ls="--", alpha=.7)
        if r == 0: ax[r, c].set_title(t, fontsize=10)
        if c == 0: ax[r, c].set_ylabel(f"{_mbd(b)} mag\n" + r"$v_\beta$ (deg/day)", fontsize=11)
        if r == len(mc_bins) - 1: ax[r, c].set_xlabel(r"$v_\lambda$ (deg/day)", fontsize=11)
    fig.colorbar(im, ax=ax[r, :], label="probability", shrink=0.85, pad=0.01)
fig.suptitle("Antisun: MBA & NEO probability -- support mask OFF vs ON", fontsize=13)
plt.show()

## 6. ROC curves — VDP vs digest2 (v5.0 Sorcha)

Completeness-vs-contamination ROC (digest2_comparison_4panel style) for the new GMM/support-masked
v5.0 tracklets: full sky plus the five antisun-distance bands. Markers = best-F1 operating point.
VDP (blue) wins near the antisun (0–20°) and the curves converge / digest2 leads at intermediate
elongations — the same story as Table~4 in the paper, and far closer than the S3M benchmark.

In [ ]:
# ===== Figure: VDP vs digest2 ROC (completeness vs contamination) -- v5.0 Sorcha =====
def roc_xy(yy, s):
    p, r, t = precision_recall_curve(yy, s)
    f1 = np.divide(2 * p * r, p + r, out=np.zeros_like(p), where=(p + r) > 0)
    bi = int(np.argmax(f1[:-1]))
    return r * 100, (1 - p) * 100, f1[bi], r[bi] * 100, (1 - p[bi]) * 100

roc_panels = [("Full sky", np.ones(len(df), bool))]
for lo, hi in [(0, 20), (20, 40), (40, 70), (70, 110), (110, 141)]:
    roc_panels.append((rf"$|\Delta\lambda_\odot| = {lo}$--${hi}^\circ$",
                       ((df.absdlon >= lo) & (df.absdlon < hi)).to_numpy()))

fig, axes = plt.subplots(2, 3, figsize=(13, 8), sharex=True, sharey=True)
for ax, (title, m) in zip(axes.ravel(), roc_panels):
    yy = y[m]
    cv, kv, fv, mcv, mkv = roc_xy(yy, df.P_NEO_vdp.values[m])
    cd, kd, fd, mcd, mkd = roc_xy(yy, df.P_NEO_d2.values[m])
    ax.plot(cv, kv, lw=2, color="tab:blue", label="VDP")
    ax.plot(cd, kd, lw=2, color="tab:orange", ls="--", label="digest2")
    ax.scatter([mcv], [mkv], color="tab:blue", s=55, zorder=5, label=fr"VDP best $F_1$={fv:.2f}")
    ax.scatter([mcd], [mkd], color="tab:orange", s=55, marker="s", zorder=5, label=fr"digest2 best $F_1$={fd:.2f}")
    ax.set_title(fr"{title}" + "\n" + fr"$N_{{\rm NEO}}$={int(yy.sum()):,}", fontsize=9)
    ax.set_xlim(0, 100); ax.set_ylim(0, 100); ax.grid(alpha=0.3)
    ax.legend(fontsize=6.5, loc="upper left")
for ax in axes[-1, :]:
    ax.set_xlabel("NEO completeness (%)")
for ax in axes[:, 0]:
    ax.set_ylabel("Contamination (%)")
fig.suptitle("VDP vs digest2 ROC -- v5.0 Sorcha (support-masked), full sky and by elongation", fontsize=12)
plt.tight_layout(); plt.show()

## 7. Mega-grid — MBA & NEO density and P(mask OFF/ON) across elongation (mag 24-25)

Everything for one magnitude bin (24-25) at once. **Rows:** MBA log density, P(MBA) mask OFF,
P(MBA) mask ON; then NEO log density, P(NEO) mask OFF, P(NEO) mask ON. **Columns:** five elongation
directions (Δλ = 0, -30, -50, -90, -120°), one per ROC band (0–20 … 110–141°). Shows how the
velocity structure and the support-mask effect evolve with elongation: P(MBA) collapses to its
sampled locus and P(NEO) brightens in the wings, most cleanly near the antisun.

In [ ]:
# ===== Mega-grid: MBA & NEO log density + P(mask OFF/ON) across elongation, mag 24-25 =====
MG_BIN = "mag24+"
MG_DIRS = [("0", "0--20", "dlon+000"), ("-30", "20--40", "dlon-030"), ("-50", "40--70", "dlon-050"),
           ("-90", "70--110", "dlon-090"), ("-120", "110--141", "dlon-120")]
mg_off = [vdp.ProbMapSet.from_npz(f"prob_maps_grid/prob_maps_grid_{d}_lat+00.npz",
                                  mask_radius_deg_per_day=float("inf")) for *_, d in MG_DIRS]
mg_on  = [vdp.ProbMapSet.from_npz(f"prob_maps_grid/prob_maps_grid_{d}_lat+00.npz",
                                  mask_radius_deg_per_day=float("inf"), support_mask_min=1) for *_, d in MG_DIRS]
mg_rows = [("MBA", "density"), ("MBA", "off"), ("MBA", "on"),
           ("NEO", "density"), ("NEO", "off"), ("NEO", "on")]
mg_rl = {("MBA", "density"): "MBA\nlog dens.", ("MBA", "off"): "MBA P\nmask OFF", ("MBA", "on"): "MBA P\nmask ON",
         ("NEO", "density"): "NEO\nlog dens.", ("NEO", "off"): "NEO P\nmask OFF", ("NEO", "on"): "NEO P\nmask ON"}
nc = len(MG_DIRS)
fig, ax = plt.subplots(len(mg_rows), nc, figsize=(2.9 * nc, 2.9 * len(mg_rows)),
                       sharex=True, sharey=True, constrained_layout=True)
for r, (pop, kind) in enumerate(mg_rows):
    rd = []
    for c in range(nc):
        if kind == "density":
            rd.append(_density_log_map(_clip(mg_off[c], mg_off[c].results[MG_BIN]["density_maps_downweighted_raw"][pop]))[0])
        elif kind == "off":
            rd.append(_clip(mg_off[c], mg_off[c].get_probability_map(MG_BIN, pop)))
        else:
            rd.append(_clip(mg_on[c], mg_on[c].get_probability_map(MG_BIN, pop)))
    if kind == "density":
        sup = [d for d in rd if np.ma.count(d) > 0]
        vmax = max(float(np.nanmax(d)) for d in sup) if sup else 1.0
        vmin = vmax - 5; cmap = _density_cmap(); cbl = r"$\log_{10} n_0$"
    else:
        vmin, vmax = 0, 1; cmap = "viridis"; cbl = "P"
    im = None
    for c in range(nc):
        a = ax[r, c]
        im = a.imshow(rd[c], origin="lower", extent=(*_VEL_LIM, *_VEL_LIM), vmin=vmin, vmax=vmax,
                      cmap=cmap, aspect="equal", interpolation="bilinear")
        a.axvline(0, color="w", lw=.5, ls="--", alpha=.6); a.axhline(0, color="w", lw=.5, ls="--", alpha=.6)
        if r == 0:
            a.set_title(fr"$\Delta\lambda={MG_DIRS[c][0]}^\circ$" + "\n" + fr"({MG_DIRS[c][1]}$^\circ$ band)", fontsize=9)
        if c == 0:
            a.set_ylabel(mg_rl[(pop, kind)] + "\n" + r"$v_\beta$", fontsize=9)
        if r == len(mg_rows) - 1:
            a.set_xlabel(r"$v_\lambda$", fontsize=9)
    fig.colorbar(im, ax=ax[r, :], fraction=.012, pad=.008, label=cbl)
fig.suptitle("MBA & NEO: log density and P (mask OFF vs ON) across elongation -- mag 24-25", fontsize=14)
plt.show()